# Notebook 02 — Preprocessing

**Milestone 2, Step 2**

Takes the raw `(T, 53, 3)` keypoint files from Notebook 01 and produces ready-to-train numpy arrays.

### Steps
1. **Gap filling** — improved missing-hand fix applied on top of the raw .npy files:
   - *Internal gaps* (between two valid detections, any length): linear interpolation
   - *Leading/trailing gaps* (before first / after last detection): edge fill with nearest frame
   - *No detection at all* (genuinely absent hand): left as zeros
2. **Validity mask** — computed from the raw zeros BEFORE filling.  
   `mask[t, j] = 1` if joint j was detected or interpolated; `0` if never detected.  
   Saved as a separate `M_*.npy` array; used as a 4th input channel to the ST-GCN.
3. **Temporal resampling** — every video resampled to T=64 frames.
4. **Torso normalization** — centre on shoulder midpoint, scale by shoulder width (per-sample, no leakage).
5. **SD splits** — built directly from the official AI4Bharat `train_test_paths` path lists (`data/official_splits/`), not the HuggingFace parquet re-packaging (which has duplicate rows and cross-split overlap — see DATA_CARD.md).
6. **Recording-session proxy splits** — 5-fold CV using rank-by-filename proxy IDs.  
   Investigation complete (June 2026): real signer labels are not publicly available in INCLUDE.  
   These folds are retained as a supplementary robustness probe only — see the signer-ID cell.

### Outputs (saved to `data/processed/`)
```
X_sd_train.npy   (N_train, 64, 53, 3)  float32   — skeleton coords
M_sd_train.npy   (N_train, 64, 53)     float32   — validity mask (0/1)
y_sd_train.npy   (N_train,)            int64
X_sd_val.npy / M_sd_val.npy / y_sd_val.npy
X_sd_test.npy / M_sd_test.npy / y_sd_test.npy
si_splits.pkl    — 5 folds, recording-session proxy (supplementary only)
label_encoder.pkl
signer_ids.csv   — proxy IDs (signer recovery not possible; see signer-ID cell)
```

In [1]:
%pip install numpy pandas pyarrow scikit-learn tqdm --quiet

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm

# ── Paths ────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("/Users/yamini/Desktop/projects/ISL PROJECT")
DATA_DIR     = PROJECT_ROOT / "data"
RAW_KP_DIR   = DATA_DIR / "raw_keypoints"
PROC_DIR     = DATA_DIR / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

T_TARGET    = 64   # target number of frames (ablation A1 will test 32/48/64/96)
N_SIGNERS   = 15   # number of signers in INCLUDE
N_FOLDS     = 5    # SI cross-validation folds
SIGNERS_PER_FOLD = N_SIGNERS // N_FOLDS   # 3 held-out signers per fold

# Joint indices for normalization (within the 53-joint array)
# POSE_INDICES = [0, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24]
# Position 3 → left shoulder  (MediaPipe landmark 11) → 53-joint index 45
# Position 4 → right shoulder (MediaPipe landmark 12) → 53-joint index 46
L_SHOULDER = 45
R_SHOULDER = 46

print("Paths OK. T_TARGET =", T_TARGET)

Paths OK. T_TARGET = 64


In [3]:
# ── Load extraction metadata ─────────────────────────────────────────────────
meta_df = pd.read_csv(DATA_DIR / "keypoints_metadata.csv")
print(f"Loaded metadata: {len(meta_df)} rows")

# rel_path comes straight from notebook 01's own scan (category/sign/[Extra/]video_id.MOV) —
# do NOT reconstruct it by concatenation here, that silently drops the nested
# "Extra/" component that 27 official videos live under.
path_to_npy  = dict(zip(meta_df["rel_path"], meta_df["out_name"]))
path_to_sign = dict(zip(meta_df["rel_path"], meta_df["sign"]))
print(f"Lookup map size: {len(path_to_npy)}")
meta_df.head(3)

Loaded metadata: 4284 rows
Lookup map size: 4284


,category,sign,video_id,video_path,rel_path,n_frames,out_name,status,miss_left_raw,miss_right_raw,miss_left_fixed,miss_right_fixed
0,Adjectives,1. loud,MVI_5177,/Users/yamini/Desktop/projects/ISL PROJECT/inc...,Adjectives/1. loud/MVI_5177.MOV,56,Adjectives__1. loud__MVI_5177.npy,skipped,NaN,NaN,NaN,NaN
1,Adjectives,1. loud,MVI_5178,/Users/yamini/Desktop/projects/ISL PROJECT/inc...,Adjectives/1. loud/MVI_5178.MOV,64,Adjectives__1. loud__MVI_5178.npy,skipped,NaN,NaN,NaN,NaN
2,Adjectives,1. loud,MVI_5179,/Users/yamini/Desktop/projects/ISL PROJECT/inc...,Adjectives/1. loud/MVI_5179.MOV,66,Adjectives__1. loud__MVI_5179.npy,skipped,NaN,NaN,NaN,NaN


In [4]:
def resample_sequence(kps, t_target=T_TARGET):
    """
    kps      : ndarray (T_src, 53, 3)
    t_target : int — desired number of frames

    Resamples to exactly t_target frames using linear interpolation.
    Works whether the video is shorter OR longer than t_target.
    """
    T_src = kps.shape[0]
    if T_src == t_target:
        return kps.astype(np.float32)
    src_idx = np.arange(T_src, dtype=np.float32)
    tgt_idx = np.linspace(0, T_src - 1, t_target, dtype=np.float32)
    out = np.zeros((t_target, 53, 3), dtype=np.float32)
    for j in range(53):
        for c in range(3):
            out[:, j, c] = np.interp(tgt_idx, src_idx, kps[:, j, c])
    return out


def normalize_skeleton(kps):
    """
    kps : ndarray (T, 53, 3)

    Per-frame torso normalization:
    - Subtract the midpoint of left and right shoulders.
    - Divide by the 2-D shoulder width (x-y distance, ignoring z).

    Falls back to the mean shoulder position across the video if shoulders
    are missing in a particular frame (pose detection failed).
    """
    result = kps.copy()
    T = kps.shape[0]

    # Compute mean shoulders for fallback (frames where pose detected)
    ls_all = kps[:, L_SHOULDER, :]   # (T, 3)
    rs_all = kps[:, R_SHOULDER, :]
    pose_ok = ~np.all(ls_all == 0, axis=1)   # frames with valid pose

    if pose_ok.any():
        ls_mean = ls_all[pose_ok].mean(axis=0)
        rs_mean = rs_all[pose_ok].mean(axis=0)
        width_mean = np.linalg.norm(ls_mean[:2] - rs_mean[:2]) + 1e-6
    else:
        # Degenerate case: no pose detected at all — just z-score
        ls_mean = rs_mean = np.zeros(3)
        width_mean = 1.0

    for t in range(T):
        ls = kps[t, L_SHOULDER, :]
        rs = kps[t, R_SHOULDER, :]

        if np.all(ls == 0) and np.all(rs == 0):
            # Pose missing this frame — use video-level mean
            center = (ls_mean + rs_mean) / 2.0
            width  = width_mean
        else:
            center = (ls + rs) / 2.0
            width  = np.linalg.norm(ls[:2] - rs[:2]) + 1e-6

        result[t] = (kps[t] - center) / width

    return result

In [5]:
def fix_missing_hand_v2(kps):
    """
    Improved missing-hand gap filler — no frame-count cap.

    kps : ndarray (T, 53, 3)

    Strategy:
      - A joint is "detected" in frame t if kps[t, j] is NOT all-zero.
      - Internal gaps (between two detected frames, any length): linear interp.
      - Leading / trailing gaps (before first / after last detection): edge fill.
      - If a joint was NEVER detected in the video: leave as zeros.

    Returns
    -------
    filled  : (T, 53, 3) float32 — gaps closed
    mask    : (T, 53)    float32 — 1 = detected or interpolated, 0 = never-detected zero
    """
    T = kps.shape[0]
    filled = kps.copy().astype(np.float32)
    mask   = np.zeros((T, 53), dtype=np.float32)

    for j in range(53):
        col = kps[:, j, :]                         # (T, 3)
        detected = ~np.all(col == 0, axis=1)       # (T,) bool

        if not detected.any():
            # Joint never seen — leave zeros, mask stays 0
            continue

        valid_frames = np.where(detected)[0]
        first, last  = valid_frames[0], valid_frames[-1]

        # Mark detected frames as valid
        mask[detected, j] = 1.0

        # Internal gaps + edge fill via np.interp (fills outside range with boundary values)
        for c in range(3):
            filled[:, j, c] = np.interp(
                np.arange(T, dtype=np.float64),
                valid_frames.astype(np.float64),
                col[valid_frames, c],
            )

        # Internal frames (first..last) are reliably interpolated → mark valid
        mask[first : last + 1, j] = 1.0
        # Leading/trailing edge-fill frames remain 0 (unreliable extrapolation)

    return filled, mask


def resample_mask(mask, t_target=T_TARGET):
    """
    Nearest-neighbour resample of a binary (T_src, 53) mask to (t_target, 53).
    Uses nearest-neighbour to avoid blurring binary values.
    """
    T_src = mask.shape[0]
    if T_src == t_target:
        return mask.astype(np.float32)
    src_idx = np.round(np.linspace(0, T_src - 1, t_target)).astype(int)
    return mask[src_idx].astype(np.float32)


In [6]:
def load_and_preprocess(npy_name):
    """
    Load raw keypoints, apply v2 gap-fill, resample to T_TARGET frames, normalize.

    Returns
    -------
    (kps, mask) where
        kps  : (T_TARGET, 53, 3) float32 — normalized skeleton coords
        mask : (T_TARGET, 53)    float32 — validity mask (1=reliable, 0=gap)
    Returns (None, None) on failure.
    """
    path = RAW_KP_DIR / npy_name
    if not path.exists():
        return None, None
    kps = np.load(str(path))
    if kps.shape[0] < 2:
        return None, None

    # 1. Improved gap fill — compute mask BEFORE resampling
    kps, mask = fix_missing_hand_v2(kps)

    # 2. Resample keypoints (linear) and mask (nearest-neighbour)
    kps  = resample_sequence(kps, T_TARGET)
    mask = resample_mask(mask, T_TARGET)

    # 3. Torso normalization
    kps = normalize_skeleton(kps)

    return kps, mask


In [7]:
# ── Fit label encoder across ALL sign classes ────────────────────────────────
all_labels = sorted(meta_df["sign"].unique())
le = LabelEncoder()
le.fit(all_labels)
print(f"Number of sign classes: {len(le.classes_)}")

with open(PROC_DIR / "label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
print("Label encoder saved.")

Number of sign classes: 262
Label encoder saved.


In [8]:
# ── Build clean SD splits directly from the OFFICIAL AI4Bharat path lists ────
#
# Ground truth: data/official_splits/{include_train,include_val,include_test}.txt
# (AI4Bharat/INCLUDE GitHub train_test_paths/) — 4,292 unique videos, zero
# cross-split overlap, verified independently (2026-08-21/22 reconciliation).
# The HuggingFace parquet used previously is a SEPARATE, corrupted re-packaging
# of this same benchmark (duplicate rows + cross-split overlap) and must not
# be used as the source of truth for splits.

OFFICIAL_DIR = DATA_DIR / "official_splits"

def load_official(name):
    return (OFFICIAL_DIR / f"include_{name}.txt").read_text().splitlines()

official_paths = {
    "train": load_official("train"),
    "val":   load_official("val"),
    "test":  load_official("test"),
}

_tr, _va, _te = (set(official_paths[s]) for s in ("train", "val", "test"))
assert not (_tr & _va) and not (_tr & _te) and not (_va & _te), \
    "official lists overlap — should never happen, re-check the source files"
print(f"Official paths: train={len(_tr)}, val={len(_va)}, test={len(_te)}, "
      f"total={len(_tr | _va | _te)}")


def build_sd_split_official(paths, split_name):
    """
    Build X, M, y arrays directly from the official path list for this split.
    A path not resolvable to a local npy file (currently only the 8
    "Second (Number)" rows, whose paths don't exist anywhere in the archive
    under any name — see DATA_CARD.md) is skipped and counted.
    """
    X_list, M_list, y_list = [], [], []
    skipped = 0
    for rel_path in tqdm(paths, desc=f"SD {split_name}"):
        npy_name = path_to_npy.get(rel_path)
        if npy_name is None:
            skipped += 1
            continue
        kps, mask = load_and_preprocess(npy_name)
        if kps is None:
            skipped += 1
            continue
        label_int = le.transform([path_to_sign[rel_path]])[0]
        X_list.append(kps)
        M_list.append(mask)
        y_list.append(label_int)
    print(f"  {split_name}: {len(X_list)} samples loaded, {skipped} not recoverable locally")
    return (np.stack(X_list).astype(np.float32),
            np.stack(M_list).astype(np.float32),
            np.array(y_list, dtype=np.int64))


X_train, M_train, y_train = build_sd_split_official(official_paths["train"], "train")
X_val,   M_val,   y_val   = build_sd_split_official(official_paths["val"],   "val")
X_test,  M_test,  y_test  = build_sd_split_official(official_paths["test"],  "test")

# ── Leakage assertions ────────────────────────────────────────────────────────
tr_npy = set(path_to_npy.get(p) for p in official_paths["train"] if path_to_npy.get(p))
va_npy = set(path_to_npy.get(p) for p in official_paths["val"]   if path_to_npy.get(p))
te_npy = set(path_to_npy.get(p) for p in official_paths["test"]  if path_to_npy.get(p))

assert len(tr_npy & te_npy) == 0, f"LEAKAGE: {len(tr_npy & te_npy)} train/test shared files"
assert len(tr_npy & va_npy) == 0, f"LEAKAGE: {len(tr_npy & va_npy)} train/val shared files"
assert len(va_npy & te_npy) == 0, f"LEAKAGE: {len(va_npy & te_npy)} val/test shared files"
print("\nLeakage check PASSED — zero shared npy files across all splits (official-path-based)")

total_unique = len(tr_npy | va_npy | te_npy)
print(f"\nOfficial SD split shapes:")
print(f"  Train : X={X_train.shape}, M={M_train.shape}, y={y_train.shape}")
print(f"  Val   : X={X_val.shape},   M={M_val.shape},   y={y_val.shape}")
print(f"  Test  : X={X_test.shape},  M={M_test.shape},  y={y_test.shape}")
print(f"  Sum   : {len(X_train)+len(X_val)+len(X_test)} samples "
      f"({total_unique} unique npy files, zero cross-split overlap, "
      f"built directly from the official AI4Bharat path lists)")

Official paths: train=3127, val=348, test=817, total=4292


SD train:   0%|          | 0/3127 [00:00<?, ?it/s]

  train: 3121 samples loaded, 6 not recoverable locally


SD val:   0%|          | 0/348 [00:00<?, ?it/s]

  val: 347 samples loaded, 1 not recoverable locally


SD test:   0%|          | 0/817 [00:00<?, ?it/s]

  test: 816 samples loaded, 1 not recoverable locally

Leakage check PASSED — zero shared npy files across all splits (official-path-based)

Official SD split shapes:
  Train : X=(3121, 64, 53, 3), M=(3121, 64, 53), y=(3121,)
  Val   : X=(347, 64, 53, 3),   M=(347, 64, 53),   y=(347,)
  Test  : X=(816, 64, 53, 3),  M=(816, 64, 53),  y=(816,)
  Sum   : 4284 samples (4284 unique npy files, zero cross-split overlap, built directly from the official AI4Bharat path lists)


In [9]:
# ── Save SD splits ───────────────────────────────────────────────────────────
np.save(str(PROC_DIR / "X_sd_train.npy"), X_train)
np.save(str(PROC_DIR / "M_sd_train.npy"), M_train)
np.save(str(PROC_DIR / "y_sd_train.npy"), y_train)

np.save(str(PROC_DIR / "X_sd_val.npy"),   X_val)
np.save(str(PROC_DIR / "M_sd_val.npy"),   M_val)
np.save(str(PROC_DIR / "y_sd_val.npy"),   y_val)

np.save(str(PROC_DIR / "X_sd_test.npy"),  X_test)
np.save(str(PROC_DIR / "M_sd_test.npy"),  M_test)
np.save(str(PROC_DIR / "y_sd_test.npy"),  y_test)

print("SD arrays saved.")
print(f"  X_sd_train : {X_train.shape}  ({X_train.nbytes / 1e6:.1f} MB)")
print(f"  M_sd_train : {M_train.shape}  ({M_train.nbytes / 1e6:.1f} MB)")


SD arrays saved.
  X_sd_train : (3121, 64, 53, 3)  (127.0 MB)
  M_sd_train : (3121, 64, 53)  (42.3 MB)


In [10]:
# ── Proxy signer IDs — INVESTIGATION COMPLETE, SIGNER IDs NOT RECOVERABLE ────
#
# ⚠  FINDING (confirmed): Real signer labels cannot be recovered from the
#    INCLUDE dataset files.  Investigation findings:
#
#    1. The Zenodo README describes "15 different word categories" — the "15"
#       is categories (Adjectives, Animals, …), NOT 15 signers.
#
#    2. MVI numbers are per-camera counters, not global identifiers.
#       461 of 3,138 unique MVI numbers appear in more than one category,
#       confirming multiple cameras were running simultaneously.  The same
#       MVI number in different categories refers to a different physical video.
#
#    3. No signer assignment file is present in the Zenodo download, the
#       HuggingFace parquet, or the AI4Bharat GitHub repo (train_test_paths/
#       files checked — only video path lists, no signer column anywhere).
#
#    4. Per-group video counts (387 → 125) and ~14 videos/sign average
#       confirm the rank-by-filename proxy does not recover real individuals.
#
#    SUPERVISOR DECISION (June 2026):
#      - Keep INCLUDE. Clean SD split is the main benchmark.
#      - AUTSL is off the table (Turkish SL, 25-joint Kinect, pipeline incompatible).
#      - SI is DROPPED as a primary evaluation claim.
#      - Stated plainly as a limitation: INCLUDE ships no signer labels; the
#        field needs signer-labelled ISL benchmarks.
#      - These proxy folds are retained as a supplementary recording-session
#        probe ONLY — never labelled signer-independent in any reported result.
#      - Headline RQ: topology ablation (single vs dual vs adaptive adjacency).
#
# The proxy IDs below are retained so NB05 can run without errors.
# SI folds produced here are for supplementary use only and must never be
# reported as cross-signer generalisation results.

N_SIGNERS = 15   # placeholder — NOT a confirmed signer count; see findings above

meta_with_si = meta_df.copy()
meta_with_si["signer_id"] = -1

for sign_class, group in meta_with_si.groupby("sign"):
    sorted_idx = group.sort_values("video_id").index
    for rank, idx in enumerate(sorted_idx):
        meta_with_si.at[idx, "signer_id"] = rank % N_SIGNERS

print("⚠  Proxy signer ID distribution (for reference only — not real signers):")
print(f"   Expected ~{len(meta_df) // N_SIGNERS} per group if balanced; actual:")
print(meta_with_si["signer_id"].value_counts().sort_index())
print()
print("   Uneven counts confirm proxy IDs do not recover real signer identities.")

meta_with_si[["category", "sign", "video_id", "out_name", "signer_id"]].to_csv(
    DATA_DIR / "signer_ids.csv", index=False
)
print("signer_ids.csv saved (proxy IDs — signer recovery not possible; see comment above).")

⚠  Proxy signer ID distribution (for reference only — not real signers):
   Expected ~285 per group if balanced; actual:
signer_id
0     387
1     387
2     384
3     384
4     380
5     325
6     270
7     262
8     230
9     229
10    228
11    228
12    228
13    227
14    135
Name: count, dtype: int64

   Uneven counts confirm proxy IDs do not recover real signer identities.
signer_ids.csv saved (proxy IDs — signer recovery not possible; see comment above).


In [11]:
# ── Build 5-fold Signer-Independent (SI) splits ──────────────────────────────
# Map out_name → signer_id
npy_to_signer = dict(zip(meta_with_si["out_name"], meta_with_si["signer_id"]))

print("Loading all keypoints for SI split construction...")
all_X, all_M, all_y, all_signer = [], [], [], []

for _, row in tqdm(meta_with_si.iterrows(), total=len(meta_with_si),
                   desc="Loading for SI"):
    npy_name = row["out_name"]
    if pd.isna(npy_name):
        continue
    kps, mask = load_and_preprocess(npy_name)
    if kps is None:
        continue
    try:
        label_int = le.transform([row["sign"]])[0]
    except Exception:
        continue
    all_X.append(kps)
    all_M.append(mask)
    all_y.append(label_int)
    all_signer.append(int(row["signer_id"]))

all_X      = np.stack(all_X).astype(np.float32)
all_M      = np.stack(all_M).astype(np.float32)
all_y      = np.array(all_y, dtype=np.int64)
all_signer = np.array(all_signer, dtype=np.int64)

print(f"Total samples for SI: {len(all_X)} (signers 0-{N_SIGNERS - 1})")


Loading all keypoints for SI split construction...


Loading for SI:   0%|          | 0/4284 [00:00<?, ?it/s]

Total samples for SI: 4284 (signers 0-14)


In [12]:
# ── 5-fold splits using proxy signer IDs — ⚠ PROVISIONAL ────────────────────
# These folds are built on proxy signer IDs (rank by filename).
# They do NOT guarantee that the same real person is absent from both
# train and test.  Results from these folds must not be reported as
# signer-independent until real signer labels are confirmed.

signer_fold = np.array([s // SIGNERS_PER_FOLD for s in range(N_SIGNERS)])

si_splits = []
for fold in range(N_FOLDS):
    held_out_signers = np.where(signer_fold == fold)[0]
    test_mask  = np.isin(all_signer, held_out_signers)
    train_mask = ~test_mask

    si_splits.append({
        "fold"     : fold,
        "held_out" : held_out_signers.tolist(),
        "X_train"  : all_X[train_mask],
        "M_train"  : all_M[train_mask],
        "y_train"  : all_y[train_mask],
        "X_test"   : all_X[test_mask],
        "M_test"   : all_M[test_mask],
        "y_test"   : all_y[test_mask],
    })
    print(f"  Fold {fold}: hold out proxy-group {held_out_signers.tolist()} "
          f"| train={train_mask.sum()}, test={test_mask.sum()}")

with open(PROC_DIR / "si_splits.pkl", "wb") as f:
    pickle.dump(si_splits, f)
print("\nSI splits saved to si_splits.pkl  (PROVISIONAL — proxy signer IDs)")

  Fold 0: hold out proxy-group [0, 1, 2] | train=3126, test=1158
  Fold 1: hold out proxy-group [3, 4, 5] | train=3195, test=1089
  Fold 2: hold out proxy-group [6, 7, 8] | train=3522, test=762


  Fold 3: hold out proxy-group [9, 10, 11] | train=3599, test=685


  Fold 4: hold out proxy-group [12, 13, 14] | train=3694, test=590



SI splits saved to si_splits.pkl  (PROVISIONAL — proxy signer IDs)


In [13]:
# ── Final summary ────────────────────────────────────────────────────────────
n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
n_total = n_train + n_val + n_test
n_official = len(_tr | _va | _te)

print("=" * 65)
print("Preprocessing complete — official-path-based splits.")
print("=" * 65)
print()
print("=== Count reconciliation (against AI4Bharat official train_test_paths) ===")
print(f"  Official unique videos (train+val+test)       : {n_official}")
print(f"  Unique .npy files on disk                      : {len(path_to_npy)}")
print(f"  SD samples (one per official path, this run)   : {n_total}  ({n_train}+{n_val}+{n_test})")
print(f"  Official paths not recoverable locally         : {n_official - n_total}  "
      f"('Second (Number)' rows — see DATA_CARD.md)")
print()
print("SD splits (built directly from official path lists — zero cross-split overlap by construction):")
print(f"  Train : {n_train:>4} samples  {X_train.shape}")
print(f"  Val   : {n_val:>4} samples  {X_val.shape}")
print(f"  Test  : {n_test:>4} samples  {X_test.shape}")
print(f"  Sum   : {n_total:>4} samples")
print()
print(f"Classes : {len(le.classes_)} (262 of 263 — 'Second (Number)' excluded;")
print(f"          those paths don't exist anywhere in the source archive; documented in DATA_CARD.md)")
print()
print(f"T_TARGET : {T_TARGET} frames")
print(f"SI folds : {N_FOLDS} folds × {SIGNERS_PER_FOLD} held-out proxy groups  "
      f"(⚠ PROVISIONAL — real signer IDs not recoverable; see signer cell above)")
print()
print(f"Mask stats (SD train):")
print(f"  Mean validity across all 53 joints : {M_train.mean():.3f}")
print(f"  ⚠ NOTE: The pooled 0.959 figure is diluted by the 11 body-pose joints")
print(f"    which are nearly always detected.  Hand joints (42 of 53) are less")
print(f"    reliable — approximately 23% of frames have at least one missing hand,")
print(f"    giving hand-joint validity ~77%.  Report hand and body validity")
print(f"    separately; do not cite 0.959 as the overall validity figure.")
print(f"  Fully valid videos: {(M_train.min(axis=(1,2))==1).sum()}/{n_train}")
print()
print("Files in data/processed/:")
for f in sorted(PROC_DIR.iterdir()):
    print(f"  {f.name:<35} {f.stat().st_size/1e6:>7.1f} MB")

Preprocessing complete — official-path-based splits.

=== Count reconciliation (against AI4Bharat official train_test_paths) ===
  Official unique videos (train+val+test)       : 4292
  Unique .npy files on disk                      : 4284
  SD samples (one per official path, this run)   : 4284  (3121+347+816)
  Official paths not recoverable locally         : 8  ('Second (Number)' rows — see DATA_CARD.md)

SD splits (built directly from official path lists — zero cross-split overlap by construction):
  Train : 3121 samples  (3121, 64, 53, 3)
  Val   :  347 samples  (347, 64, 53, 3)
  Test  :  816 samples  (816, 64, 53, 3)
  Sum   : 4284 samples

Classes : 262 (262 of 263 — 'Second (Number)' excluded;
          those paths don't exist anywhere in the source archive; documented in DATA_CARD.md)

T_TARGET : 64 frames
SI folds : 5 folds × 3 held-out proxy groups  (⚠ PROVISIONAL — real signer IDs not recoverable; see signer cell above)

Mask stats (SD train):
  Mean validity across all 53 